# Revision analyses R4 — reproducibility check of the within-cave blocks
Checks, on the real data, that the patched `BlockedWithinCaveSplitter` (the exact cell from the pipeline notebooks 001 and 002) together with `wcbg_heldout_blocks.csv` reproduces the 46 held-out blocks behind Table 2, fold by fold, in all 12 saved prediction files. No models are trained; it takes about a minute. Nothing on Drive is modified.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
import os, json, shutil, logging
from pathlib import Path
import numpy as np
import pandas as pd
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("repo_check")

_candidates = [Path(os.environ["SISAL_BASE"])] if "SISAL_BASE" in os.environ else []
_candidates += [Path("/content/drive/MyDrive/SISALv3_Speleothem"), Path("/content/drive/My Drive/SISALv3_Speleothem")]
base = next((p for p in _candidates if p.exists()), _candidates[0])
ARTIFACTS   = base / "artifacts"
WITHIN_PRED = ARTIFACTS / "20260612T233845Z" / "predictions_within"
BLOCKS_SRC  = ARTIFACTS / "revision_2026_11" / "supplement" / "wcbg_heldout_blocks.csv"
REPORT = {}
for name, p in [("base", base), ("within predictions", WITHIN_PRED), ("blocks csv", BLOCKS_SRC)]:
    print(f"{name:20s}: {p} | exists: {p.exists()}")

# The patched notebooks look for the CSV in the working directory (repo root) or the project root.
# Copy it to the Colab working directory only, so nothing on Drive changes.
shutil.copy(BLOCKS_SRC, Path.cwd() / "wcbg_heldout_blocks.csv")
PROJECT_ROOT = base

master_table = pd.read_parquet(ARTIFACTS / "_data" / "master_table_enriched.parquet")
print("master_table:", master_table.shape)

## 1 · The splitter cell exactly as it appears in the pipeline notebooks

In [ ]:
# --- 2.2 BlockedWithinCaveSplitter ------------------------------------------------
import zlib


class BlockedWithinCaveSplitter:
    """Within-entity block-gap splitter for the secondary gap-fill experiment.

    For each entity that meets `min_entity_length`, we mark a single contiguous
    test block of length `max(min_block_samples, ceil(block_fraction * n))`
    centred at a deterministic offset (different per entity, derived from
    `random_state` and `entity_id` for reproducibility). The training set for
    every fold consists of:

      - every row of every *other* entity, AND
      - the rows of *this* entity that fall outside the test block.

    This makes cubic-spline / linear interpolation a meaningful baseline (the
    interpolator can use the temporally adjacent training rows of the same
    entity to bridge the gap), while still being a real prediction problem
    (the test rows themselves are unseen during fitting).

    Reproducibility (revision, Nov 2026)
    ------------------------------------
    The original version seeded each block with Python's `hash()`, which is
    randomised per interpreter session, so reruns placed the blocks differently
    from the run behind the published tables. Two fixes:

    - `fixed_blocks`: path to `wcbg_heldout_blocks.csv` (columns entity_id,
      sample_id), exported from the published run. With it (and `sample_ids`
      aligned with the rows of X), the splitter returns exactly the published
      46 blocks, in the same fold order. This is the default below whenever the
      CSV is found.
    - Without `fixed_blocks`, block positions are seeded with `zlib.crc32`,
      which is stable across sessions and machines (but gives different blocks
      from the published run).

    Notes
    -----
    - Sequences for the BiLSTM/Transformer should be sliced *per entity*; the
      sequence-builder elsewhere in the notebook will skip windows that overlap
      the test block.
    - Returned indices index into the *position-aligned* arrays passed to
      `split(X, y, groups)`. We expect `groups` to be the entity_id column.
    """

    def __init__(
        self,
        block_fraction: float,
        min_block_samples: int,
        min_entity_length: int,
        random_state: int = 42,
        fixed_blocks=None,
        sample_ids=None,
    ):
        self.block_fraction = float(block_fraction)
        self.min_block_samples = int(min_block_samples)
        self.min_entity_length = int(min_entity_length)
        self.random_state = int(random_state)
        self.fixed_blocks = fixed_blocks
        self.sample_ids = sample_ids

    def _load_fixed(self):
        if self.fixed_blocks is None:
            return None
        fb = self.fixed_blocks if isinstance(self.fixed_blocks, pd.DataFrame) else pd.read_csv(self.fixed_blocks)
        return fb[["entity_id", "sample_id"]]

    def _seed(self, ent) -> int:
        return zlib.crc32(f"{self.random_state}|{ent}".encode()) & 0xFFFFFFFF

    def get_n_splits(self, X=None, y=None, groups=None) -> int:
        fb = self._load_fixed()
        if fb is not None:
            return int(fb["entity_id"].nunique())
        if groups is None:
            return 0
        groups = pd.Series(groups).reset_index(drop=True)
        ent_sizes = groups.value_counts()
        return int((ent_sizes >= self.min_entity_length).sum())

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("BlockedWithinCaveSplitter requires `groups` (entity_id).")
        n = len(groups)
        groups = pd.Series(groups).reset_index(drop=True)

        fb = self._load_fixed()
        if fb is not None:
            if self.sample_ids is None or len(self.sample_ids) != n:
                raise ValueError("fixed_blocks requires `sample_ids` aligned with the rows of X.")
            sid = pd.Series(np.asarray(self.sample_ids))
            for ent, g in fb.groupby("entity_id", sort=True):
                test_idx = np.where(sid.isin(set(g["sample_id"])).to_numpy())[0]
                if len(test_idx) != len(g):
                    raise ValueError(f"entity {ent}: {len(g)} block samples in fixed_blocks, {len(test_idx)} found in X")
                if not (groups.values[test_idx] == ent).all():
                    raise ValueError(f"entity {ent}: fixed-block samples belong to another entity in `groups`")
                mask = np.ones(n, dtype=bool)
                mask[test_idx] = False
                yield np.where(mask)[0], test_idx
            return

        # Eligible entities = those with enough samples to host a meaningful gap.
        eligible = (
            groups.value_counts()
                  .loc[lambda s: s >= self.min_entity_length]
                  .index
                  .tolist()
        )
        for ent in sorted(eligible):
            idx = np.where(groups.values == ent)[0]
            n_ent = len(idx)
            block_size = max(self.min_block_samples, int(np.ceil(self.block_fraction * n_ent)))
            block_size = min(block_size, n_ent - 4)  # leave at least a few train rows in this entity
            # Deterministic centre per entity, stable across sessions (crc32, not hash()).
            rs = np.random.default_rng(self._seed(ent))
            margin = max(2, n_ent // 6)
            start_lo = margin
            start_hi = max(start_lo + 1, n_ent - block_size - margin)
            start = int(rs.integers(start_lo, start_hi)) if start_hi > start_lo else start_lo
            test_local = np.arange(start, start + block_size)
            test_idx = idx[test_local]
            mask = np.ones(n, dtype=bool)
            mask[test_idx] = False
            train_idx = np.where(mask)[0]
            yield train_idx, test_idx


# Exact held-out blocks behind the published within-cave results (Table 2 of the paper).
_WCBG_CANDIDATES = [Path("wcbg_heldout_blocks.csv"), Path("data/wcbg_heldout_blocks.csv"),
                    Path("../data/wcbg_heldout_blocks.csv"), PROJECT_ROOT / "wcbg_heldout_blocks.csv",
                    PROJECT_ROOT / "data" / "wcbg_heldout_blocks.csv",
                    PROJECT_ROOT / "release_github" / "data" / "wcbg_heldout_blocks.csv"]
WCBG_BLOCKS_CSV = next((p for p in _WCBG_CANDIDATES if p.exists()), None)
if WCBG_BLOCKS_CSV is None:
    log.warning("wcbg_heldout_blocks.csv not found (looked in %s); within-cave blocks will be placed with the "
                "crc32 seed and will differ from the published run.", [str(p) for p in _WCBG_CANDIDATES])
else:
    log.info("Within-cave blocks: using the published blocks from %s", WCBG_BLOCKS_CSV)

## 2 · Published blocks: patched splitter + CSV vs the 12 saved prediction files

In [ ]:
groups  = master_table["entity_id"].values
sids    = master_table["sample_id"].values
X_dummy = np.zeros((len(master_table), 1))
sp = BlockedWithinCaveSplitter(block_fraction=0.10, min_block_samples=30, min_entity_length=80,
                               random_state=42, fixed_blocks=WCBG_BLOCKS_CSV, sample_ids=sids)
folds = [(tr, te) for tr, te in sp.split(X_dummy, groups=groups)]
REPORT["1_csv_found_by_notebook_logic"] = str(WCBG_BLOCKS_CSV)
REPORT["1_folds_from_fixed_blocks"] = f"{len(folds)} (get_n_splits = {sp.get_n_splits()})"
REPORT["1_train_test_disjoint_and_complete"] = all(len(np.intersect1d(tr, te)) == 0 and len(tr) + len(te) == len(groups) for tr, te in folds)

rows = []
for p in sorted(WITHIN_PRED.glob("*.parquet")):
    pr = pd.read_parquet(p)
    ok_fold = ok_ent = 0
    for k, (tr, te) in enumerate(folds):
        sub = pr[pr["fold"] == k]
        ok_fold += set(sids[te]) == set(sub["sample_id"])
        ok_ent  += len(sub) > 0 and int(sub["entity_id"].iloc[0]) == int(groups[te[0]])
    rows.append(dict(file=p.stem, folds_in_file=pr["fold"].nunique(), folds_same_samples=ok_fold,
                     folds_same_entity=ok_ent, all_match=(ok_fold == ok_ent == len(folds) == pr["fold"].nunique())))
REPORT["2_fixed_blocks_vs_saved_predictions"] = pd.DataFrame(rows)
print(REPORT["2_fixed_blocks_vs_saved_predictions"].to_string(index=False))

## 3 · Fallback without the CSV: crc32 placement is stable and follows the same block rule

In [ ]:
sp_c = BlockedWithinCaveSplitter(block_fraction=0.10, min_block_samples=30, min_entity_length=80, random_state=42)
a = [te for _, te in sp_c.split(X_dummy, groups=groups)]
b = [te for _, te in sp_c.split(X_dummy, groups=groups)]
REPORT["3_crc32_folds"] = len(a)
REPORT["3_crc32_repeatable"] = all(np.array_equal(x, y) for x, y in zip(a, b)) and len(a) == len(b)
REPORT["3_crc32_same_entities_as_published"] = [int(groups[t[0]]) for t in a] == [int(groups[te[0]]) for _, te in folds]
REPORT["3_crc32_same_block_sizes_as_published"] = [len(t) for t in a] == [len(te) for _, te in folds]
REPORT["3_crc32_blocks_identical_to_published (expected False)"] = sum(set(x) == set(te) for x, (_, te) in zip(a, folds))
for k in [k for k in REPORT if k.startswith("3_")]: print(k, ":", REPORT[k])

## 4 · Report — summary of all results

In [ ]:
def _fmt(v):
    if isinstance(v, pd.DataFrame): return v.to_string(index=False)
    return str(v)
_lines = ["=" * 90, "SISAL REVISION — REPRODUCIBILITY CHECK (R4)", f"generated: {pd.Timestamp.now(tz='UTC')}", "=" * 90]
for k, v in REPORT.items():
    _lines += ["", f"### {k}", _fmt(v)]
_all = (REPORT["1_train_test_disjoint_and_complete"] and REPORT["2_fixed_blocks_vs_saved_predictions"]["all_match"].all()
        and REPORT["3_crc32_repeatable"] and REPORT["3_crc32_same_entities_as_published"] and REPORT["3_crc32_same_block_sizes_as_published"])
_lines += ["", f"### VERDICT: {'PASS — patched code reproduces the published blocks' if _all else 'CHECK FAILED'}"]
print("\n".join(_lines))